# Debug Hunyuan3D — Image → 3D Mesh

This notebook isolates the Hunyuan3D mesh-generation step so you can test
it with an **existing image** (no Gemini generation needed). Each cell is
self-contained and prints diagnostics so you can pinpoint exactly where
the pipeline breaks.

## 0 — Configuration

Set your input image path and Hunyuan3D Space URL here.

In [11]:
from pathlib import Path

# ── EDIT THESE ──────────────────────────────────────────────────────────
INPUT_IMAGE = Path("../notebooks_output/create_instruction/reference.png")  # any .png/.jpg
PROMPT      = "crochet amigurumi bear"  # text caption sent alongside image
SPACE_URL   = "tencent/Hunyuan3D-2"     # HF Space or local URL like http://localhost:7860
OUTPUT_DIR  = Path("../notebooks_output/debug_hunyuan3d")
# ────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input image exists: {INPUT_IMAGE.is_file()}")
print(f"Output dir:        {OUTPUT_DIR}")

Input image exists: True
Output dir:        ../notebooks_output/debug_hunyuan3d


## 1 — Verify `gradio_client` installation

In [12]:
try:
    import gradio_client
    print(f"gradio_client version: {gradio_client.__version__}")
except ImportError:
    print("❌ gradio_client is NOT installed.")
    print("   Run:  pip install gradio_client>=1.0")
    raise

gradio_client version: 2.4.1


## 2 — Connect to the Hunyuan3D Space

In [13]:
from gradio_client import Client

print(f"Connecting to: {SPACE_URL} ...")
client = Client(SPACE_URL)
print(f"Connected.  Server info: {client.src}")

Connecting to: tencent/Hunyuan3D-2 ...
Loaded as API: https://tencent-hunyuan3d-2.hf.space
Connected.  Server info: https://tencent-hunyuan3d-2.hf.space


## 3 — Inspect available API endpoints

This shows every endpoint the Space exposes, along with parameter
names/types. Look for endpoints containing "generation", "shape",
"image_to", or "text_to".

In [14]:
try:
    api_info = client.view_api(return_format="dict", print_info=False)
except TypeError:
    api_info = client.view_api(return_format="dict")

named = api_info.get("named_endpoints", {}) if isinstance(api_info, dict) else {}
print(f"Found {len(named)} named endpoints:\n")

for ep_name, meta in sorted(named.items()):
    params = meta.get("parameters", []) if isinstance(meta, dict) else []
    param_summary = []
    for p in params:
        label = p.get("label") or p.get("parameter_name") or "?"
        ptype = p.get("python_type", {})
        if isinstance(ptype, dict):
            ptype = ptype.get("type", "?")
        param_summary.append(f"{label} ({ptype})")
    print(f"  {ep_name}")
    for ps in param_summary:
        print(f"      - {ps}")
    print()

Found 12 named endpoints:

  /generation_all
      - Text Prompt (str)
      - Image (filepath)
      - Front (filepath)
      - Back (filepath)
      - Left (filepath)
      - Right (filepath)
      - Inference Steps (float)
      - Guidance Scale (float)
      - Seed (float)
      - Octree Resolution (float)
      - Remove Background (bool)
      - Number of Chunks (float)
      - Randomize seed (bool)

  /lambda

  /lambda_1

  /lambda_2

  /lambda_3

  /lambda_4

  /lambda_5

  /lambda_6

  /on_decode_mode_change
      - Decoding Mode (Literal['Low', 'Standard', 'High'])

  /on_export_click
      - File (filepath)
      - File (filepath)
      - File Type (Literal['glb', 'obj', 'ply', 'stl'])
      - Simplify Mesh (bool)
      - Include Texture (bool)
      - Target Face Number (float)

  /on_gen_mode_change
      - Generation Mode (Literal['Turbo', 'Fast', 'Standard'])

  /shape_generation
      - Text Prompt (str)
      - Image (filepath)
      - Front (filepath)
      - Back (fi

## 4 — Filter generation-capable endpoints

Same filtering logic used in `routing.py`, so you can see exactly which
endpoints the pipeline will try and in what order.

In [15]:
def is_gen(name: str) -> bool:
    n = name.lower()
    return (
        "generation" in n or "shape" in n or "text_to" in n or "image_to" in n
    ) and ("lambda" not in n and "change" not in n and "export" not in n)

def score(name: str) -> int:
    n = name.lower()
    if "generation_all" in n:   return 0
    if "shape_generation" in n: return 1
    if "image_to" in n:         return 2
    if "text_to" in n:          return 3
    return 4

gen_endpoints = []
for ep_name, meta in named.items():
    if is_gen(ep_name):
        params = meta.get("parameters", []) if isinstance(meta, dict) else []
        gen_endpoints.append((ep_name, len(params), params))

gen_endpoints.sort(key=lambda e: score(e[0]))

if not gen_endpoints:
    print("❌ No generation endpoints found! The Space may have changed its API.")
    print("   Check the full endpoint list above and adjust the is_gen() filter.")
else:
    print(f"Found {len(gen_endpoints)} generation endpoint(s) (in priority order):\n")
    for name, nparams, params in gen_endpoints:
        labels = [p.get("label") or p.get("parameter_name") or "?" for p in params]
        print(f"  {name}  ({nparams} params: {', '.join(labels)})")

Found 2 generation endpoint(s) (in priority order):

  /generation_all  (13 params: Text Prompt, Image, Front, Back, Left, Right, Inference Steps, Guidance Scale, Seed, Octree Resolution, Remove Background, Number of Chunks, Randomize seed)
  /shape_generation  (13 params: Text Prompt, Image, Front, Back, Left, Right, Inference Steps, Guidance Scale, Seed, Octree Resolution, Remove Background, Number of Chunks, Randomize seed)


## 5 — Build arguments and call each endpoint

This cell walks through every generation endpoint (highest-priority first),
shows the exact args it would send, calls `predict()`, and prints the raw
result. It stops at the first one that returns a mesh file.

In [16]:
from gradio_client import handle_file

MESH_EXT = (".glb", ".obj", ".ply", ".stl")

def find_mesh_path(obj, depth=0):
    """Recursively search a Gradio response for a mesh file path."""
    indent = "  " * depth
    if isinstance(obj, str):
        if obj.lower().endswith(MESH_EXT):
            print(f"{indent}  -> found mesh string: {obj}")
            return obj
    elif isinstance(obj, dict):
        for key in ("path", "name", "url", "file"):
            v = obj.get(key)
            if isinstance(v, str) and v.lower().endswith(MESH_EXT):
                print(f"{indent}  -> found mesh in dict['{key}']: {v}")
                return v
        for k, v in obj.items():
            got = find_mesh_path(v, depth + 1)
            if got:
                return got
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            got = find_mesh_path(v, depth + 1)
            if got:
                return got
    return None

def fill_args(params, prompt, image_path):
    """Mirror of routing.py _fill_hunyuan_args with verbose logging."""
    args = []
    for p in params:
        label = (p.get("label") or p.get("parameter_name") or "").lower()
        ptype = ""
        if isinstance(p.get("python_type"), dict):
            ptype = p["python_type"].get("type", "")
        if any(k in label for k in ("image", "img", "input_image", "upload")):
            val = handle_file(image_path) if image_path else None
            args.append(val)
            print(f"    {label:30s} -> handle_file({image_path})")
        elif any(k in label for k in ("caption", "prompt", "text", "description")):
            args.append(prompt)
            print(f"    {label:30s} -> '{prompt}'")
        elif "seed" in label:
            args.append(1234)
            print(f"    {label:30s} -> 1234")
        elif "randomize" in label:
            args.append(True)
            print(f"    {label:30s} -> True")
        elif "rembg" in label or "remove" in label or "background" in label:
            args.append(True)
            print(f"    {label:30s} -> True (remove bg)")
        elif "step" in label:
            args.append(30)
            print(f"    {label:30s} -> 30")
        elif "guidance" in label or "cfg" in label:
            args.append(5.0)
            print(f"    {label:30s} -> 5.0")
        elif "resolution" in label or "octree" in label:
            args.append(256)
            print(f"    {label:30s} -> 256")
        elif "chunk" in label:
            args.append(8000)
            print(f"    {label:30s} -> 8000")
        elif ptype == "bool":
            args.append(False)
            print(f"    {label:30s} -> False (default bool)")
        elif ptype in ("int", "float"):
            args.append(0)
            print(f"    {label:30s} -> 0 (default num)")
        else:
            args.append(None)
            print(f"    {label:30s} -> None (unmatched, type={ptype})")
    return args


image_str = str(INPUT_IMAGE) if INPUT_IMAGE.is_file() else None
if not image_str:
    print(f"⚠️  Input image not found at {INPUT_IMAGE}")
    print("   Endpoints will receive image=None (text-only mode).\n")

mesh_path = None

for api_name, nparams, params in gen_endpoints:
    print(f"\n{'='*70}")
    print(f"Trying: {api_name}  ({nparams} params)")
    print(f"{'='*70}")
    args = fill_args(params, PROMPT, image_str)
    print(f"\n  Calling client.predict() ...")
    try:
        result = client.predict(*args, api_name=api_name)
    except Exception as exc:
        print(f"  ❌ Exception: {type(exc).__name__}: {exc}")
        continue

    print(f"  Raw result type: {type(result).__name__}")
    # Print a compact repr (truncated for readability)
    result_str = repr(result)
    if len(result_str) > 2000:
        result_str = result_str[:2000] + "  ... [truncated]"
    print(f"  Raw result:\n{result_str}\n")

    mesh_path = find_mesh_path(result)
    if mesh_path:
        print(f"  ✅ Got mesh: {mesh_path}")
        break
    else:
        print(f"  ⚠️  No mesh file found in this response.")

if not mesh_path:
    print("\n❌ None of the endpoints returned a usable mesh.")
    print("   Check the raw results above to see what the Space is returning.")


Trying: /generation_all  (13 params)
    text prompt                    -> 'crochet amigurumi bear'
    image                          -> handle_file(../notebooks_output/create_instruction/reference.png)
    front                          -> None (unmatched, type=filepath)
    back                           -> None (unmatched, type=filepath)
    left                           -> None (unmatched, type=filepath)
    right                          -> None (unmatched, type=filepath)
    inference steps                -> 30
    guidance scale                 -> 5.0
    seed                           -> 1234
    octree resolution              -> 256
    remove background              -> True (remove bg)
    number of chunks               -> 8000
    randomize seed                 -> 1234

  Calling client.predict() ...
  ❌ Exception: AppError: User is runnning out of daily ZeroGPU quotas. Visit https://huggingface.co/subscribe/pro to get more ZeroGPU quota now.

Trying: /shape_generation  (

## 6 — Load and inspect the mesh

In [17]:
import trimesh

if not mesh_path:
    print("Skipped — no mesh from previous step.")
else:
    mesh = trimesh.load(mesh_path, force="mesh")
    print(f"Vertices : {len(mesh.vertices)}")
    print(f"Faces    : {len(mesh.faces)}")
    print(f"Bounds   : {mesh.bounds.tolist()}")
    print(f"Watertight: {mesh.is_watertight}")
    print(f"Extents  : {mesh.extents.tolist()}")

    # Save a local copy
    out_glb = OUTPUT_DIR / "hunyuan3d_output.glb"
    mesh.export(out_glb)
    print(f"\nSaved to: {out_glb}")

Skipped — no mesh from previous step.


## 7 — Render multi-view projections

Same render logic used in the pipeline, displayed inline.

In [18]:
import matplotlib.pyplot as plt
from IPython.display import display

if not mesh_path:
    print("Skipped — no mesh from previous step.")
else:
    views = [("front", 0, 0), ("side", 0, 90), ("top", 90, 0), ("3/4", 25, 45)]

    fig, axes = plt.subplots(1, len(views), figsize=(5 * len(views), 5), dpi=100,
                             subplot_kw={"projection": "3d"})
    for ax, (name, elev, azim) in zip(axes, views):
        ax.plot_trisurf(
            mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
            triangles=mesh.faces,
            linewidth=0.1, edgecolor="black", color="white",
        )
        ax.view_init(elev=elev, azim=azim)
        ax.set_axis_off()
        ax.set_box_aspect((1, 1, 1))
        ax.set_title(name, fontsize=12)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "multi_view.png", bbox_inches="tight", dpi=100)
    plt.show()
    print(f"Saved to: {OUTPUT_DIR / 'multi_view.png'}")

Skipped — no mesh from previous step.


## 8 — Show input image alongside mesh

Side-by-side comparison of the reference image and the generated mesh.

In [19]:
from PIL import Image

if not INPUT_IMAGE.is_file():
    print(f"No input image at {INPUT_IMAGE}")
elif not mesh_path:
    print("No mesh to compare.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                    gridspec_kw={"width_ratios": [1, 1]})
    # Input image
    img = Image.open(INPUT_IMAGE)
    ax1.imshow(img)
    ax1.set_title("Input image", fontsize=12)
    ax1.axis("off")

    # 3D mesh (3/4 view)
    ax2.remove()
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    ax2.plot_trisurf(
        mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
        triangles=mesh.faces,
        linewidth=0.1, edgecolor="black", color="white",
    )
    ax2.view_init(elev=25, azim=45)
    ax2.set_axis_off()
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_title("Hunyuan3D mesh", fontsize=12)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "comparison.png", bbox_inches="tight", dpi=100)
    plt.show()

No mesh to compare.


## 9 — Quick troubleshooting summary

In [20]:
print("=" * 60)
print("TROUBLESHOOTING SUMMARY")
print("=" * 60)

checks = [
    ("gradio_client installed",     True),  # would have raised earlier
    ("Connected to Space",          client is not None),
    ("Generation endpoints found",  len(gen_endpoints) > 0),
    ("Input image exists",          INPUT_IMAGE.is_file()),
    ("Mesh returned",               mesh_path is not None),
]

for label, ok in checks:
    status = "✅" if ok else "❌"
    print(f"  {status}  {label}")

if mesh_path:
    print(f"\nMesh file: {mesh_path}")
    print(f"Vertices:  {len(mesh.vertices)}")
    print(f"Faces:     {len(mesh.faces)}")
    print("\nThe Hunyuan3D connection is working correctly.")
else:
    print("\nCommon fixes:")
    if not gen_endpoints:
        print("  - The Space API may have changed. Check cell 3 for the")
        print("    actual endpoint names and update the is_gen() filter.")
    print("  - The HF Space may be asleep. Open it in a browser first:")
    print(f"    https://huggingface.co/spaces/{SPACE_URL}")
    print("  - Try a local Hunyuan3D server: set SPACE_URL = 'http://localhost:7860'")
    print("  - Check cell 5 for the raw response — the mesh might be")
    print("    returned in an unexpected format.")

TROUBLESHOOTING SUMMARY
  ✅  gradio_client installed
  ✅  Connected to Space
  ✅  Generation endpoints found
  ✅  Input image exists
  ❌  Mesh returned

Common fixes:
  - The HF Space may be asleep. Open it in a browser first:
    https://huggingface.co/spaces/tencent/Hunyuan3D-2
  - Try a local Hunyuan3D server: set SPACE_URL = 'http://localhost:7860'
  - Check cell 5 for the raw response — the mesh might be
    returned in an unexpected format.
